<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/05_Productivity_Classification_CHL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================
# STEP 31.9P-16R
# PRODUCTIVITY THRESHOLD DEFINITION
#
# PURPOSE:
# Define Low / Medium / High CHL productivity thresholds using ONLY
# the training CHL distribution.
#
# IMPORTANT METHODOLOGICAL RULES:
# - Training data ONLY
# - Validation data NOT loaded
# - Test data NOT loaded
# - No model retraining
# - No model modification
# - Thresholds are calculated before classification
#
# Productivity classes:
#   LOW    : CHL < 33rd percentile
#   MEDIUM : 33rd <= CHL <= 66th percentile
#   HIGH   : CHL > 66th percentile
# ======================================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive


print("=" * 70)
print("STEP 31.9P-16R")
print("PRODUCTIVITY THRESHOLD DEFINITION")
print("=" * 70)


# ======================================================================
# 1. MOUNT GOOGLE DRIVE
# ======================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ======================================================================
# 2. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESSING_DIR = (
    PROJECT_DIR / "05_model_preprocessing"
)

OUTPUT_DIR = (
    PROJECT_DIR /
    "06_models" /
    "18_productivity_classification"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


TRAIN_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_train.csv"
)

THRESHOLD_FILE = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.csv"
)

THRESHOLD_JSON = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.json"
)


print("\n" + "=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print("\nProject:")
print(PROJECT_DIR)

print("\nTraining dataset:")
print(TRAIN_FILE)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

print("\nTraining file exists:", TRAIN_FILE.exists())

if not TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Required training file not found: {TRAIN_FILE}"
    )

print("\n✓ Training dataset found.")


# ======================================================================
# 4. EXPLICIT DATA-SAFETY CHECK
# ======================================================================

VALIDATION_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_validation.csv"
)

TEST_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_test.csv"
)

print("\n" + "=" * 70)
print("DATA USAGE SAFETY")
print("=" * 70)

print("""
TRAINING DATA : WILL BE USED
VALIDATION    : NOT LOADED
TEST DATA     : NOT LOADED

Thresholds will be calculated exclusively from the training
chlorophyll_a distribution.
""")

# Deliberately DO NOT load validation or test data.


# ======================================================================
# 5. LOAD TRAINING DATA
# ======================================================================

print("=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(TRAIN_FILE)

print("\nTraining shape:")
print(train_df.shape)

print("\nTraining rows:", len(train_df))


# ======================================================================
# 6. CHECK TARGET COLUMN
# ======================================================================

TARGET_COLUMN = "chlorophyll_a"

print("\n" + "=" * 70)
print("TARGET COLUMN CHECK")
print("=" * 70)

print("\nTarget column:", TARGET_COLUMN)

if TARGET_COLUMN not in train_df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' not found in training data."
    )

print("✓ chlorophyll_a column found.")


# ======================================================================
# 7. TARGET SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING CHL SAFETY CHECK")
print("=" * 70)

chl = pd.to_numeric(
    train_df[TARGET_COLUMN],
    errors="coerce"
)

nan_count = chl.isna().sum()
inf_count = np.isinf(chl.to_numpy()).sum()

print("\nCHL NaN values:", nan_count)
print("CHL infinite values:", inf_count)

if nan_count > 0:
    raise ValueError(
        "Training chlorophyll_a contains NaN values."
    )

if inf_count > 0:
    raise ValueError(
        "Training chlorophyll_a contains infinite values."
    )

if len(chl) == 0:
    raise ValueError(
        "No valid training CHL observations found."
    )

print("\n✓ Training CHL values passed safety checks.")


# ======================================================================
# 8. TRAINING CHL DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("TRAINING CHL DISTRIBUTION")
print("=" * 70)

print("\nNumber of CHL observations:", len(chl))

print(
    f"Minimum CHL       : {chl.min():.6f} mg/m³"
)

print(
    f"Maximum CHL       : {chl.max():.6f} mg/m³"
)

print(
    f"Mean CHL          : {chl.mean():.6f} mg/m³"
)

print(
    f"Median CHL        : {chl.median():.6f} mg/m³"
)


# ======================================================================
# 9. CALCULATE PRODUCTIVITY THRESHOLDS
#
# IMPORTANT:
# These percentiles are calculated ONLY from TRAINING CHL.
# ======================================================================

print("\n" + "=" * 70)
print("CALCULATING PRODUCTIVITY THRESHOLDS")
print("=" * 70)

LOW_MEDIUM_PERCENTILE = 33
MEDIUM_HIGH_PERCENTILE = 66

low_medium_threshold = np.percentile(
    chl,
    LOW_MEDIUM_PERCENTILE
)

medium_high_threshold = np.percentile(
    chl,
    MEDIUM_HIGH_PERCENTILE
)

print(
    f"\n33rd percentile threshold : "
    f"{low_medium_threshold:.6f} mg/m³"
)

print(
    f"66th percentile threshold : "
    f"{medium_high_threshold:.6f} mg/m³"
)


# ======================================================================
# 10. VERIFY THRESHOLD ORDER
# ======================================================================

print("\n" + "=" * 70)
print("THRESHOLD LOGIC CHECK")
print("=" * 70)

if not low_medium_threshold < medium_high_threshold:
    raise ValueError(
        "Invalid productivity thresholds: "
        "33rd percentile must be below 66th percentile."
    )

print(
    "\n✓ 33rd percentile < 66th percentile."
)

print(
    "✓ Three ordered productivity regions can be defined."
)


# ======================================================================
# 11. DEFINE PRODUCTIVITY CLASS RULES
# ======================================================================

print("\n" + "=" * 70)
print("PRODUCTIVITY CLASS DEFINITIONS")
print("=" * 70)

print("""
LOW:
    chlorophyll_a < 33rd percentile

MEDIUM:
    33rd percentile <= chlorophyll_a <= 66th percentile

HIGH:
    chlorophyll_a > 66th percentile
""")


# ======================================================================
# 12. APPLY CLASS RULES TO TRAINING DATA ONLY
#
# This is ONLY a diagnostic check to verify class balance.
# It does NOT involve validation or test data.
# ======================================================================

def classify_productivity(chl_value):
    if chl_value < low_medium_threshold:
        return "Low"
    elif chl_value <= medium_high_threshold:
        return "Medium"
    else:
        return "High"


training_classes = chl.apply(
    classify_productivity
)

class_counts = (
    training_classes
    .value_counts()
    .reindex(
        ["Low", "Medium", "High"],
        fill_value=0
    )
)

class_percentages = (
    training_classes
    .value_counts(normalize=True)
    .reindex(
        ["Low", "Medium", "High"],
        fill_value=0
    ) * 100
)


print("\n" + "=" * 70)
print("TRAINING PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

distribution_df = pd.DataFrame({
    "productivity_class": class_counts.index,
    "observations": class_counts.values,
    "percentage": class_percentages.values
})

print(
    distribution_df.to_string(
        index=False
    )
)


# ======================================================================
# 13. SAVE THRESHOLDS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING PRODUCTIVITY THRESHOLDS")
print("=" * 70)

threshold_df = pd.DataFrame({
    "threshold_type": [
        "low_medium_boundary",
        "medium_high_boundary"
    ],
    "percentile": [
        LOW_MEDIUM_PERCENTILE,
        MEDIUM_HIGH_PERCENTILE
    ],
    "threshold_mg_m3": [
        low_medium_threshold,
        medium_high_threshold
    ],
    "source_dataset": [
        "training_only",
        "training_only"
    ],
    "target_variable": [
        TARGET_COLUMN,
        TARGET_COLUMN
    ]
})

threshold_df.to_csv(
    THRESHOLD_FILE,
    index=False
)

print("\nSaved:")
print(THRESHOLD_FILE)


# ======================================================================
# 14. SAVE JSON CONFIGURATION
# ======================================================================

threshold_config = {
    "method": "training_distribution_percentile",
    "target_variable": TARGET_COLUMN,
    "source_dataset": "final_model_ready_train.csv",
    "validation_data_used": False,
    "test_data_used": False,
    "low_medium_percentile": LOW_MEDIUM_PERCENTILE,
    "medium_high_percentile": MEDIUM_HIGH_PERCENTILE,
    "low_medium_threshold_mg_m3": float(
        low_medium_threshold
    ),
    "medium_high_threshold_mg_m3": float(
        medium_high_threshold
    ),
    "classification_rules": {
        "Low": (
            f"CHL < {low_medium_threshold:.10f}"
        ),
        "Medium": (
            f"{low_medium_threshold:.10f} <= CHL "
            f"<= {medium_high_threshold:.10f}"
        ),
        "High": (
            f"CHL > {medium_high_threshold:.10f}"
        )
    }
}

with open(
    THRESHOLD_JSON,
    "w"
) as f:
    json.dump(
        threshold_config,
        f,
        indent=4
    )

print("\nSaved:")
print(THRESHOLD_JSON)


# ======================================================================
# 15. FINAL SAFETY VERIFICATION
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16R THRESHOLD SAFETY CHECK")
print("=" * 70)

print(
    "\nTraining data used        : TRUE"
)

print(
    "Validation data loaded    : FALSE"
)

print(
    "Test data loaded          : FALSE"
)

print(
    "Test data used            : FALSE"
)

print(
    "Model retraining          : FALSE"
)

print(
    "Model modification        : FALSE"
)

print(
    "Threshold source          : TRAINING ONLY"
)

print(
    f"33rd percentile           : "
    f"{low_medium_threshold:.6f} mg/m³"
)

print(
    f"66th percentile           : "
    f"{medium_high_threshold:.6f} mg/m³"
)

print("\n✓ Thresholds derived exclusively from training data.")
print("✓ No validation information used.")
print("✓ No test information used.")
print("✓ No model was retrained.")
print("✓ No existing model was modified.")
print("✓ Productivity thresholds saved successfully.")


# ======================================================================
# 16. COMPLETE
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-16R — THRESHOLD DEFINITION COMPLETE")
print("=" * 70)

print("""
✓ Training CHL distribution analysed.
✓ 33rd percentile calculated.
✓ 66th percentile calculated.
✓ Low / Medium / High boundaries defined.
✓ Training class distribution verified.
✓ Threshold CSV saved.
✓ Threshold JSON saved.
✓ Validation data NOT loaded.
✓ Test data NOT loaded.
✓ Test data NOT used.
""")

STEP 31.9P-16R
PRODUCTIVITY THRESHOLD DEFINITION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PROJECT PATHS

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Training dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

Output directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification

REQUIRED FILE CHECK

Training file exists: True

✓ Training dataset found.

DATA USAGE SAFETY

TRAINING DATA : WILL BE USED
VALIDATION    : NOT LOADED
TEST DATA     : NOT LOADED

Thresholds will be calculated exclusively from the training
chlorophyll_a distribution.

LOADING TRAINING DATA

Training shape:
(736950, 75)

Training rows: 736950

TARGET COLUMN CHECK

Target column: chlorophyll_a
✓ chlorophyll_a column found.

TRAINING CHL SAFETY CHECK

CHL NaN values: 0
CHL infinite values: 0

✓ Training CHL values 

In [2]:
# ======================================================================
# STEP 31.9P-16R-FIX
# FINAL XGBOOST PRODUCTIVITY CLASSIFICATION
# CORRECTED THRESHOLD READING
# ======================================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("STEP 31.9P-16R-FIX")
print("FINAL XGBOOST PRODUCTIVITY CLASSIFICATION")
print("=" * 70)


# ======================================================================
# 1. DRIVE
# ======================================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ======================================================================
# 2. PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

OUTPUT_DIR = (
    PROJECT_DIR /
    "06_models" /
    "18_productivity_classification"
)

THRESHOLD_FILE = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.csv"
)

THRESHOLD_JSON = (
    OUTPUT_DIR /
    "productivity_thresholds_training_only.json"
)

TEST_PREDICTIONS_FILE = (
    PROJECT_DIR /
    "06_models" /
    "02_XGBoost" /
    "final_selected_test_evaluation" /
    "xgboost_selected_final_test_predictions.csv"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

for file_path in [
    THRESHOLD_FILE,
    THRESHOLD_JSON,
    TEST_PREDICTIONS_FILE
]:

    print(
        f"\n{file_path}"
    )

    print(
        f"Exists: {file_path.exists()}"
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )

print("\n✓ All required files found.")


# ======================================================================
# 4. METHODOLOGICAL SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("METHODOLOGICAL SAFETY")
print("=" * 70)

print("""
Threshold source:
    TRAINING ONLY

Validation data:
    NOT LOADED

XGBoost test predictions:
    ALREADY GENERATED
    USED ONLY FOR CLASSIFICATION

XGBoost retraining:
    NOT PERFORMED

XGBoost modification:
    NOT PERFORMED

Threshold recalculation:
    NOT PERFORMED
""")


# ======================================================================
# 5. LOAD TRAINING-ONLY THRESHOLDS
# ======================================================================

print("\n" + "=" * 70)
print("LOADING TRAINING-ONLY PRODUCTIVITY THRESHOLDS")
print("=" * 70)

threshold_df = pd.read_csv(
    THRESHOLD_FILE
)

print("\nThreshold file contents:")
print(
    threshold_df.to_string(
        index=False
    )
)


# ======================================================================
# 6. CORRECT THRESHOLD EXTRACTION
# ======================================================================
#
# IMPORTANT:
#
# DO NOT use the "percentile" column as the threshold.
#
# percentile = 33 / 66
#
# threshold_mg_m3 = actual CHL boundary
#
# We explicitly select threshold_mg_m3 here.
# ======================================================================

required_threshold_columns = [
    "threshold_type",
    "percentile",
    "threshold_mg_m3",
    "source_dataset",
    "target_variable"
]

missing_threshold_columns = [
    col
    for col in required_threshold_columns
    if col not in threshold_df.columns
]

if missing_threshold_columns:
    raise ValueError(
        "Missing threshold columns: "
        f"{missing_threshold_columns}"
    )


# Verify source is training only

if not (
    threshold_df["source_dataset"]
    .astype(str)
    .str.lower()
    .eq("training_only")
    .all()
):

    raise ValueError(
        "Threshold file contains values not explicitly "
        "marked as training_only."
    )


# Extract actual CHL thresholds

low_medium_row = threshold_df[
    threshold_df["threshold_type"]
    .astype(str)
    .str.lower()
    .eq("low_medium_boundary")
]

medium_high_row = threshold_df[
    threshold_df["threshold_type"]
    .astype(str)
    .str.lower()
    .eq("medium_high_boundary")
]


if len(low_medium_row) != 1:
    raise ValueError(
        "Could not uniquely identify "
        "low_medium_boundary."
    )

if len(medium_high_row) != 1:
    raise ValueError(
        "Could not uniquely identify "
        "medium_high_boundary."
    )


lower_threshold = float(
    low_medium_row.iloc[0]["threshold_mg_m3"]
)

upper_threshold = float(
    medium_high_row.iloc[0]["threshold_mg_m3"]
)


# ======================================================================
# 7. VERIFY CORRECT THRESHOLDS
# ======================================================================

print("\n" + "=" * 70)
print("LOCKED TRAINING-ONLY THRESHOLDS")
print("=" * 70)

print(
    f"\nLow / Medium boundary : "
    f"{lower_threshold:.6f} mg/m³"
)

print(
    f"Medium / High boundary: "
    f"{upper_threshold:.6f} mg/m³"
)

# Expected values from the completed threshold-definition step

expected_lower = 0.212671
expected_upper = 0.433100

if not np.isclose(
    lower_threshold,
    expected_lower,
    atol=1e-5
):

    raise ValueError(
        f"Unexpected lower threshold: "
        f"{lower_threshold}"
    )

if not np.isclose(
    upper_threshold,
    expected_upper,
    atol=1e-5
):

    raise ValueError(
        f"Unexpected upper threshold: "
        f"{upper_threshold}"
    )

if lower_threshold >= upper_threshold:

    raise ValueError(
        "Invalid threshold ordering."
    )

print(
    "\n✓ Correct CHL threshold values confirmed."
)

print(
    "✓ Percentile numbers were NOT used as CHL thresholds."
)

print(
    "✓ Thresholds originate from training data only."
)


# ======================================================================
# 8. LOAD FINAL XGBOOST TEST PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL XGBOOST TEST PREDICTIONS")
print("=" * 70)

pred_df = pd.read_csv(
    TEST_PREDICTIONS_FILE
)

print(
    f"\nTest prediction shape: "
    f"{pred_df.shape}"
)


required_prediction_columns = [
    "actual_chlorophyll_a",
    "predicted_chlorophyll_a"
]

missing_prediction_columns = [
    col
    for col in required_prediction_columns
    if col not in pred_df.columns
]

if missing_prediction_columns:

    raise ValueError(
        "Missing prediction columns: "
        f"{missing_prediction_columns}"
    )

print(
    "\n✓ Required actual and predicted CHL columns found."
)


# ======================================================================
# 9. SAFETY CHECK
# ======================================================================

actual_chl = pd.to_numeric(
    pred_df["actual_chlorophyll_a"],
    errors="coerce"
)

predicted_chl = pd.to_numeric(
    pred_df["predicted_chlorophyll_a"],
    errors="coerce"
)

print("\n" + "=" * 70)
print("CHL SAFETY CHECK")
print("=" * 70)

print(
    f"\nActual CHL NaN       : "
    f"{actual_chl.isna().sum()}"
)

print(
    f"Predicted CHL NaN   : "
    f"{predicted_chl.isna().sum()}"
)

print(
    f"Actual CHL infinite : "
    f"{np.isinf(actual_chl).sum()}"
)

print(
    f"Predicted CHL infinite: "
    f"{np.isinf(predicted_chl).sum()}"
)

if actual_chl.isna().any():
    raise ValueError(
        "Actual CHL contains NaN."
    )

if predicted_chl.isna().any():
    raise ValueError(
        "Predicted CHL contains NaN."
    )

if np.isinf(actual_chl).any():
    raise ValueError(
        "Actual CHL contains infinite values."
    )

if np.isinf(predicted_chl).any():
    raise ValueError(
        "Predicted CHL contains infinite values."
    )

print(
    "\n✓ CHL prediction values are valid."
)


# ======================================================================
# 10. CLASSIFICATION FUNCTION
# ======================================================================

def classify_productivity(chl):

    if chl < lower_threshold:

        return "Low"

    elif chl <= upper_threshold:

        return "Medium"

    else:

        return "High"


# ======================================================================
# 11. APPLY PRODUCTIVITY CLASSES
# ======================================================================

print("\n" + "=" * 70)
print("APPLYING PRODUCTIVITY CLASSIFICATION")
print("=" * 70)

pred_df[
    "actual_productivity_class"
] = actual_chl.apply(
    classify_productivity
)

pred_df[
    "predicted_productivity_class"
] = predicted_chl.apply(
    classify_productivity
)

print("""
Classification rules:

Low:
    CHL < 0.212671 mg/m³

Medium:
    0.212671 <= CHL <= 0.433100 mg/m³

High:
    CHL > 0.433100 mg/m³
""")


# ======================================================================
# 12. ACTUAL CLASS DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("ACTUAL PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

class_order = [
    "Low",
    "Medium",
    "High"
]

actual_distribution = (
    pred_df[
        "actual_productivity_class"
    ]
    .value_counts()
    .reindex(
        class_order,
        fill_value=0
    )
    .reset_index()
)

actual_distribution.columns = [
    "productivity_class",
    "observations"
]

actual_distribution[
    "percentage"
] = (
    actual_distribution[
        "observations"
    ]
    / len(pred_df)
    * 100
)

print(
    actual_distribution.to_string(
        index=False
    )
)


# ======================================================================
# 13. PREDICTED CLASS DISTRIBUTION
# ======================================================================

print("\n" + "=" * 70)
print("PREDICTED PRODUCTIVITY CLASS DISTRIBUTION")
print("=" * 70)

predicted_distribution = (
    pred_df[
        "predicted_productivity_class"
    ]
    .value_counts()
    .reindex(
        class_order,
        fill_value=0
    )
    .reset_index()
)

predicted_distribution.columns = [
    "productivity_class",
    "observations"
]

predicted_distribution[
    "percentage"
] = (
    predicted_distribution[
        "observations"
    ]
    / len(pred_df)
    * 100
)

print(
    predicted_distribution.to_string(
        index=False
    )
)


# ======================================================================
# 14. CONFUSION MATRIX
# ======================================================================

print("\n" + "=" * 70)
print("PRODUCTIVITY CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(
    pred_df[
        "actual_productivity_class"
    ],
    pred_df[
        "predicted_productivity_class"
    ],
    labels=class_order
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual Low",
        "Actual Medium",
        "Actual High"
    ],
    columns=[
        "Predicted Low",
        "Predicted Medium",
        "Predicted High"
    ]
)

print(
    cm_df.to_string()
)


# ======================================================================
# 15. OVERALL CLASSIFICATION METRICS
# ======================================================================

actual_classes = pred_df[
    "actual_productivity_class"
]

predicted_classes = pred_df[
    "predicted_productivity_class"
]

accuracy = accuracy_score(
    actual_classes,
    predicted_classes
)

macro_precision = precision_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="macro",
    zero_division=0
)

weighted_precision = precision_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="weighted",
    zero_division=0
)

weighted_recall = recall_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="weighted",
    zero_division=0
)

weighted_f1 = f1_score(
    actual_classes,
    predicted_classes,
    labels=class_order,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("OVERALL PRODUCTIVITY CLASSIFICATION PERFORMANCE")
print("=" * 70)

print(
    f"\nAccuracy           : {accuracy:.6f}"
)

print(
    f"Macro Precision    : {macro_precision:.6f}"
)

print(
    f"Macro Recall       : {macro_recall:.6f}"
)

print(
    f"Macro F1           : {macro_f1:.6f}"
)

print(
    f"Weighted Precision : {weighted_precision:.6f}"
)

print(
    f"Weighted Recall    : {weighted_recall:.6f}"
)

print(
    f"Weighted F1        : {weighted_f1:.6f}"
)


# ======================================================================
# 16. CLASS-WISE METRICS
# ======================================================================

print("\n" + "=" * 70)
print("CLASS-WISE PRODUCTIVITY PERFORMANCE")
print("=" * 70)

report = classification_report(
    actual_classes,
    predicted_classes,
    labels=class_order,
    target_names=class_order,
    output_dict=True,
    zero_division=0
)

class_metrics = []

for class_name in class_order:

    class_metrics.append({

        "productivity_class":
            class_name,

        "precision":
            report[class_name]["precision"],

        "recall":
            report[class_name]["recall"],

        "f1_score":
            report[class_name]["f1-score"],

        "support":
            report[class_name]["support"]
    })

class_metrics_df = pd.DataFrame(
    class_metrics
)

print(
    class_metrics_df.to_string(
        index=False
    )
)


# ======================================================================
# 17. CLASSIFICATION AGREEMENT
# ======================================================================

agreement = (
    actual_classes ==
    predicted_classes
)

agreement_count = agreement.sum()

agreement_percentage = (
    agreement_count
    / len(pred_df)
    * 100
)

print("\n" + "=" * 70)
print("PRODUCTIVITY CLASSIFICATION AGREEMENT")
print("=" * 70)

print(
    f"\nCorrect classifications:"
    f" {agreement_count:,}"
)

print(
    f"Total observations:"
    f" {len(pred_df):,}"
)

print(
    f"Agreement:"
    f" {agreement_percentage:.4f}%"
)


# ======================================================================
# 18. SAVE CLASSIFIED PREDICTIONS
# ======================================================================

classified_predictions_file = (
    OUTPUT_DIR /
    "xgboost_final_test_productivity_predictions.csv"
)

pred_df.to_csv(
    classified_predictions_file,
    index=False
)


# ======================================================================
# 19. SAVE CONFUSION MATRIX
# ======================================================================

confusion_matrix_file = (
    OUTPUT_DIR /
    "productivity_confusion_matrix.csv"
)

cm_df.to_csv(
    confusion_matrix_file
)


# ======================================================================
# 20. SAVE CLASS-WISE METRICS
# ======================================================================

class_metrics_file = (
    OUTPUT_DIR /
    "productivity_classwise_metrics.csv"
)

class_metrics_df.to_csv(
    class_metrics_file,
    index=False
)


# ======================================================================
# 21. SAVE OVERALL METRICS
# ======================================================================

overall_metrics = pd.DataFrame([

    {
        "metric": "accuracy",
        "value": accuracy
    },

    {
        "metric": "macro_precision",
        "value": macro_precision
    },

    {
        "metric": "macro_recall",
        "value": macro_recall
    },

    {
        "metric": "macro_f1",
        "value": macro_f1
    },

    {
        "metric": "weighted_precision",
        "value": weighted_precision
    },

    {
        "metric": "weighted_recall",
        "value": weighted_recall
    },

    {
        "metric": "weighted_f1",
        "value": weighted_f1
    }
])

overall_metrics_file = (
    OUTPUT_DIR /
    "productivity_classification_metrics.csv"
)

overall_metrics.to_csv(
    overall_metrics_file,
    index=False
)


# ======================================================================
# 22. SAVE DISTRIBUTIONS
# ======================================================================

actual_distribution.to_csv(
    OUTPUT_DIR /
    "actual_productivity_distribution.csv",
    index=False
)

predicted_distribution.to_csv(
    OUTPUT_DIR /
    "predicted_productivity_distribution.csv",
    index=False
)


# ======================================================================
# 23. SAVE SUMMARY
# ======================================================================

summary = {

    "step":
        "31.9P-16R-FIX",

    "threshold_source":
        "training_only",

    "low_medium_threshold_mg_m3":
        lower_threshold,

    "medium_high_threshold_mg_m3":
        upper_threshold,

    "test_observations":
        int(len(pred_df)),

    "accuracy":
        float(accuracy),

    "macro_precision":
        float(macro_precision),

    "macro_recall":
        float(macro_recall),

    "macro_f1":
        float(macro_f1),

    "weighted_precision":
        float(weighted_precision),

    "weighted_recall":
        float(weighted_recall),

    "weighted_f1":
        float(weighted_f1),

    "model_retrained":
        False,

    "model_modified":
        False,

    "thresholds_recalculated":
        False,

    "validation_data_loaded":
        False,

    "test_predictions_source":
        str(TEST_PREDICTIONS_FILE)
}

summary_file = (
    OUTPUT_DIR /
    "productivity_classification_summary.json"
)

with open(
    summary_file,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ======================================================================
# 24. FINAL SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FINAL 31.9P-16R-FIX SAFETY CHECK")
print("=" * 70)

print(
    "\nCorrect threshold column used : TRUE"
)

print(
    "Threshold source              : TRAINING ONLY"
)

print(
    "Low/Medium threshold          : "
    f"{lower_threshold:.6f} mg/m³"
)

print(
    "Medium/High threshold         : "
    f"{upper_threshold:.6f} mg/m³"
)

print(
    "Validation data loaded        : FALSE"
)

print(
    "Final test predictions used   : TRUE"
)

print(
    "XGBoost retraining            : FALSE"
)

print(
    "XGBoost modification          : FALSE"
)

print(
    "\n✓ Correct thresholds applied."
)

print(
    "✓ Test predictions were not regenerated."
)

print(
    "✓ XGBoost model was not modified."
)

print(
    "✓ Classification results saved."
)


# ======================================================================
# 25. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-16R-FIX COMPLETE")
print("=" * 70)

print(
    f"""
Training-derived productivity boundaries:

LOW:
    CHL < {lower_threshold:.6f} mg/m³

MEDIUM:
    {lower_threshold:.6f} <= CHL <=
    {upper_threshold:.6f} mg/m³

HIGH:
    CHL > {upper_threshold:.6f} mg/m³

Accuracy:
    {accuracy:.6f}

Macro F1:
    {macro_f1:.6f}

Weighted F1:
    {weighted_f1:.6f}
"""
)

print(
    "Next stage:"
)

print(
    "31.9P-16S — SHAP Explainability"
)

print("=" * 70)

STEP 31.9P-16R-FIX
FINAL XGBOOST PRODUCTIVITY CLASSIFICATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

REQUIRED FILE CHECK

/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification/productivity_thresholds_training_only.csv
Exists: True

/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/18_productivity_classification/productivity_thresholds_training_only.json
Exists: True

/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/final_selected_test_evaluation/xgboost_selected_final_test_predictions.csv
Exists: True

✓ All required files found.

METHODOLOGICAL SAFETY

Threshold source:
    TRAINING ONLY

Validation data:
    NOT LOADED

XGBoost test predictions:
    ALREADY GENERATED
    USED ONLY FOR CLASSIFICATION

XGBoost retraining:
    NOT PERFORMED

XGBoost modification:
    NOT PERFORMED

Threshold recalculation:
    NOT PERFORMED


LOAD